# Elliptic-curve discrete logarithm

Recover the order-five logarithm with the original sampled table-driven workflow. Build the original synthesis-only arithmetic workflow using six reversible Kaliski rounds. Verify its emitted prefix on every scalar pair, then compare the complex result after the inverse Fourier suffix.

Set the original curve, generator, seed and target.

In [ ]:
import examples.synthesis.algorithms.number_theory_and_cryptography.elliptic_curves.elliptic_curve_discrete_log.overview.main as report
import examples.synthesis.algorithms.number_theory_and_cryptography.elliptic_curves.elliptic_curve_discrete_log.overview.workflow as workflow
from examples.synthesis.algorithms.number_theory_and_cryptography.elliptic_curves.elliptic_curve_discrete_log.curve import (
    add,
)
from examples.synthesis.algorithms.number_theory_and_cryptography.elliptic_curves.elliptic_curve_discrete_log.main import (
    build_program,
)
from flagquantum.compiler import ResourceEstimate, estimate_resources

print('Curve: y² = x³ + 5x + 4 mod 7')
print(workflow.GENERATOR, workflow.ORDER, workflow.SEED, workflow.TARGET)

Use two three-bit scalar registers and a six-bit point register.

In [ ]:
assert tuple(map(len, (workflow.X1, workflow.X2, workflow.POINT_X, workflow.POINT_Y))) == (3, 3, 3, 3)

Control point translations with the three bits of each scalar.

In [ ]:
print('Controlled powers:', (1, 2, 4))

Double the generator classically to supply each controlled addend.

In [ ]:
twice = add(workflow.GENERATOR, workflow.GENERATOR)
assert twice is not None
four_times = add(twice, twice)
print(workflow.GENERATOR, twice, four_times)

Use the table-driven point translation for the sampled route.

In [ ]:
mock_point = workflow.sampled_point_add

Pass the source generator to the original-orbit translation.

In [ ]:
assert add(workflow.SEED, workflow.GENERATOR) == (4, 5)

Synthesize and sample the native point addition.

In [ ]:
point, point_resources = mock_point()
report.print_resources('Mock point gates', point_resources)

Check the sample against the affine curve equation.

In [ ]:
assert point == (4, 5)
print('Sampled point sum:', point)

Compose both scalar accumulations and the inverse QFTs.

In [ ]:
build_mock = build_program

Synthesize the complete mock period-finding program.

In [ ]:
mock_program = build_mock()
mock_resources = mock_program.report['resources']
assert isinstance(mock_resources, ResourceEstimate)

Read the actual instruction mix from the synthesized program.

In [ ]:
print('Mock instruction counts:', mock_resources.operation_counts)

Keep the native circuit depth in the resource report.

In [ ]:
print('Mock instruction depth:', mock_resources.depth)

Draw 2,048 joint scalar-and-point samples.

In [ ]:
rows, period_resources, mock_error = workflow.sampled_period_finding(mock_program)
report.print_resources('Mock ECDLP gates', period_resources)
print('Mock absolute state error:', mock_error)

Sort by count and inspect the ten highest-frequency outcomes.

In [ ]:
report.print_frequencies(rows)

Round measured fixed-point scalars to order-five residues.

In [ ]:
rounded_left = [round(row.left * workflow.ORDER / 8) for row in rows]
rounded_right = [round(row.right * workflow.ORDER / 8) for row in rows]
print(list(zip(rounded_left[:10], rounded_right[:10], strict=True)))

Skip rows with noninvertible rounded right residues.

In [ ]:
eligible = [row for row in rows if row.logarithm() is not None]
assert eligible

Verify the ten highest-frequency eligible logarithms.

In [ ]:
report.print_logarithms(rows)

Use six reversible Kaliski rounds and clear the slope from the output.

In [ ]:
build_arithmetic_point = workflow.point_add_circuit

Build and inspect the arithmetic point-addition Circuit.

In [ ]:
point_circuit = build_arithmetic_point()
print('Kaliski point sum:', workflow.checked_point_add(point_circuit))
report.print_resources('Kaliski point gates', estimate_resources(point_circuit))
del point_circuit

Compose both arithmetic scalar multiplications in one Circuit.

In [ ]:
build_arithmetic_period = workflow.period_finding_circuit

Inspect the arithmetic Circuit and verify its complete coherent result.

In [ ]:
period_circuit = build_arithmetic_period()
report.print_resources('Kaliski ECDLP gates', estimate_resources(period_circuit))
print('Kaliski absolute state error:', workflow.checked_period_finding(period_circuit))